<a href="https://colab.research.google.com/github/malikjunaidgul/DataAnalytics/blob/DataLoading/Dataloading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Tabular and Semi Structured Data
Purpose: download and load CSV, Excel, JSON, XML into pandas for cleaning and modeling.

In [2]:
# load_tabular_and_json.py
import os
import requests
import zipfile
import io
import pandas as pd
import json
import xml.etree.ElementTree as ET

os.makedirs("data/tabular", exist_ok=True)

# 1) Download CSV (example UCI student)
csv_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
r = requests.get(csv_url)
r.raise_for_status()
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall("data/tabular/uci_student")

# 2) Load CSV/Excel
df_csv = pd.read_csv("data/tabular/uci_student/student-mat.csv", sep=";")

# Fix for the TypeError:
# Separate numeric and non-numeric columns
numeric_cols = df_csv.select_dtypes(include=['number']).columns
object_cols = df_csv.select_dtypes(include=['object']).columns

# Fill NaN in numeric columns with their mean
df_csv[numeric_cols] = df_csv[numeric_cols].fillna(df_csv[numeric_cols].mean())

# Fill NaN in object columns with their mode
for col in object_cols:
    df_csv[col] = df_csv[col].fillna(df_csv[col].mode()[0])

# 3) Download JSON (example placeholder)
json_url = "https://raw.githubusercontent.com/typicode/demo/master/db.json"
r = requests.get(json_url)
r.raise_for_status()
data = r.json()
df_json = pd.json_normalize(data)

# 4) Load XML string example
xml_str = """<root><item><id>1</id><value>A</value></item></root>"""
root = ET.fromstring(xml_str)
rows = []
for item in root.findall("item"):
    rows.append({child.tag: child.text for child in item})
df_xml = pd.DataFrame(rows)

print("CSV rows:", len(df_csv), "JSON rows:", len(df_json), "XML rows:", len(df_xml))

CSV rows: 395 JSON rows: 1 XML rows: 1


In [4]:
# robust_tabular_loader.py
import pandas as pd
import numpy as np
from pathlib import Path

def load_tabular(path, sep=None, parse_dates=None):
    path = Path(path)
    if path.suffix in [".csv", ".txt"]:
        df = pd.read_csv(path, sep=sep)
    elif path.suffix in [".xls", ".xlsx"]:
        df = pd.read_excel(path)
    else:
        raise ValueError("Unsupported file type: " + path.suffix)

    # 1) Convert obvious numeric columns (safe coercion)
    for col in df.columns:
        if df[col].dtype == object:
            coerced = pd.to_numeric(df[col], errors="coerce")
            # if many values converted, keep numeric
            if coerced.notna().sum() > 0.5 * len(df):
                df[col] = coerced

    # 2) Fill numeric columns with mean (numeric_only)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    if len(numeric_cols):
        df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

    # 3) Fill categorical/object columns with mode or placeholder
    obj_cols = df.select_dtypes(include=["object", "category"]).columns
    for c in obj_cols:
        if df[c].isna().any():
            try:
                mode = df[c].mode(dropna=True)
                fill = mode.iloc[0] if len(mode) else "missing"
            except Exception:
                fill = "missing"
            df[c] = df[c].fillna(fill)

    return df

# Example
df = load_tabular("data/tabular/uci_student/student-mat.csv")
print(df.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   school      395 non-null    object
 1   sex         395 non-null    object
 2   age         395 non-null    int64 
 3   address     395 non-null    object
 4   famsize     395 non-null    object
 5   Pstatus     395 non-null    object
 6   Medu        395 non-null    int64 
 7   Fedu        395 non-null    int64 
 8   Mjob        395 non-null    object
 9   Fjob        395 non-null    object
 10  reason      395 non-null    object
 11  guardian    395 non-null    object
 12  traveltime  395 non-null    int64 
 13  studytime   395 non-null    int64 
 14  failures    395 non-null    int64 
 15  schoolsup   395 non-null    object
 16  famsup      395 non-null    object
 17  paid        395 non-null    object
 18  activities  395 non-null    object
 19  nursery     395 non-null    object
 20  higher    

/tmp/ipython-input-193/2806891078.py:9: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(path, sep=sep)


JSON and XML loader (semi‑structured)

In [5]:
# semi_structured_loader.py
import json
import pandas as pd
import xml.etree.ElementTree as ET
from pandas import json_normalize

def load_json_file(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # If top-level is list of records
    if isinstance(data, list):
        return json_normalize(data)
    # If nested dict, try to find records key
    for k, v in data.items():
        if isinstance(v, list):
            return json_normalize(v)
    return json_normalize(data)

def load_xml_string(xml_str):
    root = ET.fromstring(xml_str)
    rows = []
    for item in root:
        rows.append({child.tag: child.text for child in item})
    return pd.DataFrame(rows)
